# Alfido Tech | Sales Performance Analysis

**Dataset:** [Kaggle Superstore Sales](https://www.kaggle.com/datasets/bhanupratapbiswas/superstore-sales)

## Purpose
Explore revenue, profitability, products, categories, regions and time trends. This notebook cleans the data, computes KPIs, visualizes performance, and exports analysis tables.

## Setup
1. Download the dataset from Kaggle (respect its license/terms).
2. Put the CSV or Excel file in the same directory as this notebook.
3. Run all cells from top to bottom.

**Interpretation note:** The source is a Superstore dataset, not Alfido Tech's internal transactions. Use findings as an analytical case study and validate any proposed action against Alfido Tech's own customer, margin, inventory and channel data. Conversion rate is only calculated if a valid funnel denominator is present.


In [ ]:
# Alfido Tech | Sales Performance Analysis
# Dataset: Kaggle Superstore Sales
# Run this notebook after placing the downloaded dataset CSV/XLSX in the same folder.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DATA_DIR = Path(".")
candidate_files = list(DATA_DIR.glob("*.csv")) + list(DATA_DIR.glob("*.xlsx")) + list(DATA_DIR.glob("*.xls"))
candidate_files = [p for p in candidate_files if "cleaned_sales" not in p.name.lower()]
if not candidate_files:
    raise FileNotFoundError("Place the Kaggle Superstore CSV or Excel file beside this notebook, then rerun.")
file_path = candidate_files[0]
df = pd.read_csv(file_path) if file_path.suffix.lower()==".csv" else pd.read_excel(file_path)
print("Loaded:", file_path.name, "| Shape:", df.shape)
display(df.head())

# Standardize column names and normalize common variants.
def norm(s):
    return str(s).strip().lower().replace("/", "_").replace("-", "_").replace(" ", "_")
df.columns = [norm(c) for c in df.columns]
aliases = {
    "orderdate":"order_date", "shipdate":"ship_date", "sub_category":"sub_category",
    "subcategory":"sub_category", "productname":"product_name", "productid":"product_id",
    "orderid":"order_id", "customerid":"customer_id", "rowid":"row_id",
    "sales_amount":"sales", "revenue":"sales", "quantity_sold":"quantity"
}
df = df.rename(columns={c:aliases.get(c,c) for c in df.columns})

# Data quality review
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(Markdown("## Data quality report"))
display(quality)

# Parse date fields and clean whitespace.
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].map(lambda x: x.strip() if isinstance(x,str) else x)
for c in ["order_date","ship_date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

for c in ["sales","profit","quantity","discount"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

rows_before = len(df)
df = df.drop_duplicates()
duplicate_rows_removed = rows_before-len(df)

# Filter invalid key metrics conservatively; retain negative profit because it represents loss.
if "sales" in df.columns:
    df = df[df["sales"].notna() & (df["sales"] >= 0)]
if "quantity" in df.columns:
    df = df[df["quantity"].notna() & (df["quantity"] > 0)]

# Feature engineering
if "order_date" in df.columns:
    df["year"] = df.order_date.dt.year
    df["quarter"] = df.order_date.dt.to_period("Q").astype(str)
    df["month_num"] = df.order_date.dt.month
    df["month"] = df.order_date.dt.strftime("%b")
    df["year_month"] = df.order_date.dt.to_period("M").astype(str)
if {"profit","sales"}.issubset(df.columns):
    df["line_profit_margin"] = np.where(df.sales != 0, df.profit/df.sales, np.nan)
if {"ship_date","order_date"}.issubset(df.columns):
    df["days_to_ship"] = (df.ship_date-df.order_date).dt.days

# Order-level aggregation prevents counting each line as a separate order for AOV.
order_count = df["order_id"].nunique() if "order_id" in df.columns else np.nan
revenue = df["sales"].sum() if "sales" in df.columns else np.nan
profit = df["profit"].sum() if "profit" in df.columns else np.nan
units = df["quantity"].sum() if "quantity" in df.columns else np.nan
aov = revenue/order_count if pd.notna(order_count) and order_count else np.nan
margin = profit/revenue if pd.notna(revenue) and revenue else np.nan
kpis = pd.DataFrame([
    ("Revenue", revenue), ("Profit", profit), ("Profit margin", margin),
    ("Distinct orders", order_count), ("Average order value", aov), ("Units sold", units),
    ("Duplicate rows removed", duplicate_rows_removed)
], columns=["KPI","Value"])
display(Markdown("## Executive KPIs"))
display(kpis)

if "order_id" in df.columns and "sales" in df.columns:
    order_sales = df.groupby("order_id", as_index=False).sales.sum()
    assert order_sales.order_id.nunique() == len(order_sales)
    display(Markdown("**AOV** is total line-item sales divided by distinct order IDs."))

# Conversion metric availability check
conversion_fields = [c for c in df.columns if any(k in c for k in ["visitor","session","lead","impression","click","conversion"])]
display(Markdown("## Conversion data availability"))
if conversion_fields:
    print("Potential funnel fields found:", conversion_fields, "| Define the event and denominator before calculating conversion.")
else:
    print("Conversion rate unavailable: no visitor/session/lead/funnel denominator found in the loaded data.")

def bar_chart(data, x, y, title, xlabel=None, ylabel=None, top=None, ascending=False):
    plot = data.sort_values(y, ascending=ascending)
    if top:
        plot = plot.head(top)
    plt.figure(figsize=(11,5))
    plt.bar(plot[x].astype(str), plot[y])
    plt.title(title); plt.xlabel(xlabel or x); plt.ylabel(ylabel or y)
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

# Regional performance
if "region" in df.columns:
    region = df.groupby("region", dropna=False).agg(
        revenue=("sales","sum") if "sales" in df else ("region","size"),
        profit=("profit","sum") if "profit" in df else ("region","size"),
        orders=("order_id","nunique") if "order_id" in df else ("region","size")
    ).reset_index()
    display(region.sort_values("revenue", ascending=False))
    bar_chart(region, "region", "revenue", "Revenue by Region")

# Category / subcategory
for dim in ["category","sub_category"]:
    if dim in df.columns:
        agg = df.groupby(dim, dropna=False).agg(
            revenue=("sales","sum") if "sales" in df else (dim,"size"),
            profit=("profit","sum") if "profit" in df else (dim,"size")
        ).reset_index().sort_values("revenue", ascending=False)
        display(agg)
        bar_chart(agg, dim, "revenue", f"Revenue by {dim.replace('_',' ').title()}")

# Product performance, both revenue and profit
if "product_name" in df.columns:
    metrics = {"revenue":("sales","sum")} if "sales" in df else {}
    if "profit" in df: metrics["profit"]=("profit","sum")
    products = df.groupby("product_name", dropna=False).agg(**metrics).reset_index()
    if "sales" in df:
        display(Markdown("### Top 10 products by revenue"))
        display(products.nlargest(10,"revenue"))
        bar_chart(products.nlargest(10,"revenue"), "product_name", "revenue", "Top 10 Products by Revenue")
        display(Markdown("### Bottom 10 products by revenue"))
        display(products.nsmallest(10,"revenue"))
    if "profit" in df:
        display(Markdown("### Lowest-profit products (including loss-making items)"))
        display(products.nsmallest(10,"profit"))
        bar_chart(products.nsmallest(10,"profit"), "product_name", "profit", "10 Lowest-Profit Products", ascending=True)

# Time trends and seasonality
if "order_date" in df.columns and "sales" in df.columns:
    monthly = df.groupby("year_month", as_index=False).sales.sum()
    plt.figure(figsize=(12,5)); plt.plot(monthly.year_month, monthly.sales, marker="o")
    plt.title("Monthly Revenue Trend"); plt.xlabel("Year-Month"); plt.ylabel("Revenue")
    plt.xticks(rotation=60, ha="right"); plt.tight_layout(); plt.show()
    seasonal = df.groupby("month_num", as_index=False).sales.mean().merge(
        df[["month_num","month"]].drop_duplicates(), on="month_num", how="left"
    ).sort_values("month_num")
    display(Markdown("### Average sales by calendar month (across available years)"))
    display(seasonal[["month","sales"]])
    plt.figure(figsize=(10,4)); plt.bar(seasonal.month, seasonal.sales)
    plt.title("Average Revenue by Calendar Month"); plt.xlabel("Month"); plt.ylabel("Average monthly line sales")
    plt.tight_layout(); plt.show()

# Discount analysis
if {"discount","sales","profit"}.issubset(df.columns):
    discount = df.groupby("discount", as_index=False).agg(revenue=("sales","sum"), profit=("profit","sum"), rows=("discount","size"))
    display(Markdown("### Discount vs. revenue/profit (descriptive, not causal)"))
    display(discount.sort_values("discount"))
    plt.figure(figsize=(8,5)); plt.scatter(discount.discount, discount.profit)
    plt.title("Observed Discount vs Total Profit"); plt.xlabel("Discount"); plt.ylabel("Profit")
    plt.tight_layout(); plt.show()

# Save prepared dataset and analytical tables
df.to_csv("cleaned_superstore_sales.csv", index=False)
if "region" in df.columns:
    region.to_csv("regional_performance.csv", index=False)
if "product_name" in df.columns:
    products.to_csv("product_performance.csv", index=False)
if "order_date" in df.columns and "sales" in df.columns:
    monthly.to_csv("monthly_revenue.csv", index=False)
print("Exported cleaned data and available summary tables to the notebook folder.")